In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    precision_recall_curve, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from lightgbm import LGBMClassifier
import joblib

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

print("OK")

OK


In [2]:
df = pd.read_parquet('../data/processed.parquet')
print(f"Размер: {df.shape}")
print(f"Диалогов: {df['dialogue_id'].nunique()}")
print(f"Доля is_end=1: {df['is_end'].mean():.3f}")
df.head()

Размер: (153983, 11)
Диалогов: 2000
Доля is_end=1: 0.088


,dialogue_id,turn_id,prefix_text,is_end,length_words,length_chars,last_word,ends_with_punct,has_question_word,last_pos,ends_with_prep
0,PMUL4398.json,0,i,0,1,1,i,0,0,PRON,0
1,PMUL4398.json,0,i need,0,2,6,need,0,0,VERB,0
2,PMUL4398.json,0,i need a,0,3,8,a,0,0,DET,0
3,PMUL4398.json,0,i need a place,0,4,14,place,0,0,NOUN,0
4,PMUL4398.json,0,i need a place to,0,5,17,to,0,0,PART,0


In [3]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(df, groups=df['dialogue_id']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"Train: {len(train_df):,} строк, {train_df['dialogue_id'].nunique()} диалогов")
print(f"Test:  {len(test_df):,} строк, {test_df['dialogue_id'].nunique()} диалогов")
print()

# Проверка: нет пересечений по dialogue_id
overlap = set(train_df['dialogue_id']) & set(test_df['dialogue_id'])
print(f"Пересечение диалогов: {len(overlap)}  (должно быть 0)")

# Проверка: баланс классов сохраняется в обоих сплитах
print(f"\nДоля is_end=1 в train: {train_df['is_end'].mean():.3f}")
print(f"Доля is_end=1 в test:  {test_df['is_end'].mean():.3f}")

Train: 122,667 строк, 1600 диалогов
Test:  31,316 строк, 400 диалогов

Пересечение диалогов: 0  (должно быть 0)

Доля is_end=1 в train: 0.088
Доля is_end=1 в test:  0.087


In [4]:
# Числовые и бинарные признаки (берем как есть)
NUMERIC_FEATURES = ['length_words', 'length_chars', 'ends_with_punct', 
                    'ends_with_prep', 'has_question_word']

# Категориальные (нужен one-hot encoding)
CATEGORICAL_FEATURES = ['last_pos']

# last_word оставим на потом (там много уникальных значений, нужен target encoding)

# Целевая переменная
y_train = train_df['is_end'].values
y_test = test_df['is_end'].values

# Признаки
X_train_raw = train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
X_test_raw = test_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]

print(f"X_train: {X_train_raw.shape}")
print(f"X_test:  {X_test_raw.shape}")
print(f"\nТипы признаков:")
print(X_train_raw.dtypes)
print(f"\nУникальные POS-теги: {X_train_raw['last_pos'].nunique()}")
print(X_train_raw['last_pos'].value_counts())


X_train: (122667, 6)
X_test:  (31316, 6)

Типы признаков:
length_words         int64
length_chars         int64
ends_with_punct      int64
ends_with_prep       int64
has_question_word    int64
last_pos               str
dtype: object

Уникальные POS-теги: 17
last_pos
NOUN     18731
PRON     16661
VERB     16270
DET      12248
ADP      11921
AUX      10277
PROPN     7077
INTJ      6390
ADJ       4765
ADV       4673
PART      4173
NUM       3952
SCONJ     2781
CCONJ     2541
X          105
PUNCT       92
SYM         10
Name: count, dtype: int64


In [5]:
# OneHotEncoder превращает категорию "ADP" в столбец где 1 если ADP, иначе 0
# handle_unknown='ignore' — если в тесте появится POS которого не было в train, не падать
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Обучаем на train, применяем на оба сплита
ohe.fit(X_train_raw[CATEGORICAL_FEATURES])

# Получаем имена новых столбцов
ohe_columns = ohe.get_feature_names_out(CATEGORICAL_FEATURES)
print(f"Создано {len(ohe_columns)} OHE-столбцов:")
print(list(ohe_columns))

# Применяем
X_train_ohe = pd.DataFrame(
    ohe.transform(X_train_raw[CATEGORICAL_FEATURES]),
    columns=ohe_columns,
    index=X_train_raw.index
)
X_test_ohe = pd.DataFrame(
    ohe.transform(X_test_raw[CATEGORICAL_FEATURES]),
    columns=ohe_columns,
    index=X_test_raw.index
)

# Собираем финальный X
X_train = pd.concat([X_train_raw[NUMERIC_FEATURES], X_train_ohe], axis=1)
X_test = pd.concat([X_test_raw[NUMERIC_FEATURES], X_test_ohe], axis=1)

print(f"\nИтоговая форма X_train: {X_train.shape}")
print(f"Столбцы: {list(X_train.columns)}")
X_train.head()

Создано 17 OHE-столбцов:
['last_pos_ADJ', 'last_pos_ADP', 'last_pos_ADV', 'last_pos_AUX', 'last_pos_CCONJ', 'last_pos_DET', 'last_pos_INTJ', 'last_pos_NOUN', 'last_pos_NUM', 'last_pos_PART', 'last_pos_PRON', 'last_pos_PROPN', 'last_pos_PUNCT', 'last_pos_SCONJ', 'last_pos_SYM', 'last_pos_VERB', 'last_pos_X']

Итоговая форма X_train: (122667, 22)
Столбцы: ['length_words', 'length_chars', 'ends_with_punct', 'ends_with_prep', 'has_question_word', 'last_pos_ADJ', 'last_pos_ADP', 'last_pos_ADV', 'last_pos_AUX', 'last_pos_CCONJ', 'last_pos_DET', 'last_pos_INTJ', 'last_pos_NOUN', 'last_pos_NUM', 'last_pos_PART', 'last_pos_PRON', 'last_pos_PROPN', 'last_pos_PUNCT', 'last_pos_SCONJ', 'last_pos_SYM', 'last_pos_VERB', 'last_pos_X']


,length_words,length_chars,ends_with_punct,ends_with_prep,has_question_word,last_pos_ADJ,last_pos_ADP,last_pos_ADV,last_pos_AUX,last_pos_CCONJ,last_pos_DET,last_pos_INTJ,last_pos_NOUN,last_pos_NUM,last_pos_PART,last_pos_PRON,last_pos_PROPN,last_pos_PUNCT,last_pos_SCONJ,last_pos_SYM,last_pos_VERB,last_pos_X
82,1,5,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
83,2,10,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
84,3,12,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
85,4,15,0,0,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
86,5,23,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
